# Advanced Drone Technology Project
## Introduction to Guidance and Control Using Python Simulations

This notebook contains the completed simulations for Task 1 (Drone PID Altitude Control), Task 2 (Boat Guidance and Path Tracking), and the Creative Bonus Task (Drone Figure-Eight Trajectory).

## Task 1: Understanding PID Controller Design Using Drone Altitude Control

The drone attempts to reach and hold a desired altitude of 10 m. A wind disturbance is introduced after t = 6 s (no-wind zone is 0-6 s). The PID gains below were tuned by first adjusting P, then D, then I, aiming for a fast response, minimum overshoot, reduced oscillation, and good disturbance rejection.

In [ ]:
"""
Task 1: Understanding PID Controller Design Using Drone Altitude Control
-------------------------------------------------------------------------
The drone attempts to reach and hold a desired altitude using a PID
controller. A wind disturbance is switched on after t = 6 s so that the
effect of the disturbance on stability / tracking can be observed.

Run this file directly:
    python task1_drone_pid.py
It will save 'pid_tuning_result.png' in the current folder.
"""

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Simulation parameters
# ---------------------------------------------------------------
dt = 0.01                 # time step (s)
t_end = 20.0               # total simulation time (s)
t = np.arange(0, t_end, dt)

mass = 1.0                 # kg  (simplified drone mass)
g = 9.81                   # m/s^2

target_altitude = 10.0     # m, desired altitude

# Wind disturbance settings
wind_start_time = 6.0       # s, "no wind zone" is 0-6 s
wind_disturbance = 3.0      # m/s^2 equivalent extra downward/side force after 6s

# ---------------------------------------------------------------
# PID gains  (tuned by hand: P first, then D, then I)
# ---------------------------------------------------------------
Kp = 6.0
Kd = 4.0
Ki = 0.5


def simulate(Kp, Ki, Kd, wind_mag):
    """Run the altitude-hold simulation and return time history arrays."""
    altitude = 0.0
    velocity = 0.0
    integral_error = 0.0
    prev_error = target_altitude - altitude

    altitude_history = []
    error_history = []

    for time_step in t:
        error = target_altitude - altitude
        integral_error += error * dt
        derivative_error = (error - prev_error) / dt
        prev_error = error

        # PID control output -> commanded thrust acceleration
        control = Kp * error + Ki * integral_error + Kd * derivative_error

        # Total acceleration = thrust (control), with gravity already
        # compensated by the hover baseline, minus the wind disturbance
        wind_force = wind_mag if time_step >= wind_start_time else 0.0
        acceleration = control - wind_force

        velocity += acceleration * dt
        altitude += velocity * dt

        altitude_history.append(altitude)
        error_history.append(error)

    return np.array(altitude_history), np.array(error_history)


if __name__ == "__main__":
    altitude_history, error_history = simulate(Kp, Ki, Kd, wind_disturbance)

    plt.figure(figsize=(9, 5))
    plt.plot(t, altitude_history, label="Drone Altitude", color="tab:blue")
    plt.axhline(target_altitude, color="black", linestyle="--", linewidth=1, label="Target Altitude")
    plt.axvline(wind_start_time, color="red", linestyle=":", linewidth=1.5, label="Wind Disturbance Introduced (t=6s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Altitude (m)")
    plt.title(f"PID Controller Tuning Result (Kp={Kp}, Ki={Ki}, Kd={Kd})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("pid_tuning_result.png", dpi=150)
    print("Saved pid_tuning_result.png")

    # Simple performance metrics printed for the report
    overshoot = max(altitude_history) - target_altitude
    settle_band = 0.05 * target_altitude
    print(f"Max overshoot: {overshoot:.3f} m")


## Task 2: Understanding Guidance and Path Tracking Using an Autonomous Boat

The boat uses a Line-Of-Sight (LOS) style guidance law with a P+D heading controller to follow a predefined curved path. Two scenarios are simulated: with water current disturbance and without (ideal, current_x = current_y = 0).

In [ ]:
"""
Task 2: Understanding Guidance and Path Tracking Using an Autonomous Boat
--------------------------------------------------------------------------
The boat tries to follow a predefined trajectory (blue dotted line) using a
simple Line-Of-Sight (LOS) guidance law with a P+D heading controller.
Water current is modelled as a constant disturbance (current_x, current_y)
added directly to the boat's velocity.

Run this file directly:
    python task2_boat_guidance.py
It will save two plots:
    boat_guidance_with_current.png
    boat_guidance_without_current.png
"""

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Desired path (a gentle curve) - this is the "blue dotted line"
# ---------------------------------------------------------------
path_t = np.linspace(0, 40, 800)
path_x = path_t
path_y = 4.0 * np.sin(0.25 * path_t)

# ---------------------------------------------------------------
# Controller gains (tuned using the slider-style search suggested
# in the assignment: kp first, then kd)
# ---------------------------------------------------------------
kp = 1.8
kd = 0.6

dt = 0.05
sim_time = 42.0
speed = 1.2          # constant forward speed of the boat (m/s)
lookahead = 6         # index-based lookahead along the path array


def wrap_angle(angle):
    """Wrap an angle to the range [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def simulate_boat(current_x, current_y):
    x, y, theta = 0.0, 0.0, 0.0
    prev_heading_error = 0.0

    xs, ys = [x], [y]
    steps = int(sim_time / dt)

    for i in range(steps):
        # find nearest path point ahead of the boat, then look ahead further
        dists = (path_x - x) ** 2 + (path_y - y) ** 2
        nearest_idx = np.argmin(dists)
        target_idx = min(nearest_idx + lookahead, len(path_x) - 1)

        target_x = path_x[target_idx]
        target_y = path_y[target_idx]

        desired_heading = np.arctan2(target_y - y, target_x - x)
        heading_error = wrap_angle(desired_heading - theta)

        d_error = (heading_error - prev_heading_error) / dt
        prev_heading_error = heading_error

        omega = kp * heading_error + kd * d_error
        theta += omega * dt

        # boat kinematics, current acts as an additive drift
        x += (speed * np.cos(theta) + current_x) * dt
        y += (speed * np.sin(theta) + current_y) * dt

        xs.append(x)
        ys.append(y)

        if nearest_idx >= len(path_x) - 2:
            break

    return np.array(xs), np.array(ys)


def plot_result(xs, ys, current_x, current_y, filename, title):
    plt.figure(figsize=(9, 5))
    plt.plot(path_x, path_y, "b--", label="Desired Path", linewidth=1.5)
    plt.plot(xs, ys, color="tab:orange", label="Boat Trajectory", linewidth=1.8)
    plt.scatter([xs[0]], [ys[0]], color="green", zorder=5, label="Start")
    plt.xlabel("X position (m)")
    plt.ylabel("Y position (m)")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis("equal")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    print(f"Saved {filename}")


if __name__ == "__main__":
    # Case 1: WITH disturbance / current
    current_x, current_y = 0.3, 0.2
    xs, ys = simulate_boat(current_x, current_y)
    plot_result(
        xs, ys, current_x, current_y,
        "boat_guidance_with_current.png",
        f"Boat Guidance WITH Current (kp={kp}, kd={kd}, current=({current_x},{current_y}))",
    )
    tracking_error = np.mean(np.sqrt((np.interp(xs, path_x, path_y) - ys) ** 2))
    print(f"With current -> mean tracking deviation approx: {tracking_error:.3f} m")

    # Case 2: WITHOUT disturbance / current (ideal conditions, as required)
    current_x, current_y = 0, 0
    xs, ys = simulate_boat(current_x, current_y)
    plot_result(
        xs, ys, current_x, current_y,
        "boat_guidance_without_current.png",
        f"Boat Guidance WITHOUT Current (kp={kp}, kd={kd}, current=(0,0))",
    )
    tracking_error = np.mean(np.sqrt((np.interp(xs, path_x, path_y) - ys) ** 2))
    print(f"Without current -> mean tracking deviation approx: {tracking_error:.3f} m")


## Creative Bonus Task: Drone Figure-Eight Trajectory

The drone tracks a lemniscate (figure-eight) trajectory using an independent PD position controller on the x and y axes, with a constant wind disturbance enabled to test robustness.

In [ ]:
"""
Creative Bonus Task: Drone Figure-Eight Trajectory Tracking
-------------------------------------------------------------
The desired path is a lemniscate (figure-eight) defined by parametric
equations. A simple PD position controller (independently on x and y)
drives the drone to follow the moving target point along the figure
eight, and wind disturbance can optionally be enabled.

Run this file directly:
    python bonus_figure_eight_drone.py
It will save 'figure_eight_drone_tracking.png'.
"""

import numpy as np
import matplotlib.pyplot as plt

dt = 0.01
t_end = 20.0
t = np.arange(0, t_end, dt)

A = 5.0        # amplitude (m)
omega = 0.4    # angular speed of the figure-eight (rad/s)

# Parametric equations of a figure-eight (Lemniscate of Gerono)
target_x = A * np.sin(omega * t)
target_y = A * np.sin(omega * t) * np.cos(omega * t)

# PD controller gains for x and y position loops
Kp = 8.0
Kd = 5.0

wind_enabled = True
wind_x, wind_y = 0.4, -0.3   # constant wind disturbance (m/s^2)


def simulate():
    x, y = 0.0, 0.0
    vx, vy = 0.0, 0.0
    prev_ex, prev_ey = target_x[0] - x, target_y[0] - y

    xs, ys = [], []

    for i, current_t in enumerate(t):
        ex = target_x[i] - x
        ey = target_y[i] - y

        dex = (ex - prev_ex) / dt
        dey = (ey - prev_ey) / dt
        prev_ex, prev_ey = ex, ey

        ax = Kp * ex + Kd * dex
        ay = Kp * ey + Kd * dey

        if wind_enabled:
            ax += wind_x
            ay += wind_y

        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt

        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)


if __name__ == "__main__":
    xs, ys = simulate()

    plt.figure(figsize=(7, 7))
    plt.plot(target_x, target_y, "b--", label="Desired Figure-Eight Path", linewidth=1.5)
    plt.plot(xs, ys, color="tab:orange", label="Drone Trajectory", linewidth=1.8)
    plt.scatter([0], [0], color="green", zorder=5, label="Start")
    plt.xlabel("X position (m)")
    plt.ylabel("Y position (m)")
    plt.title("Bonus Task: Drone Following a Figure-Eight Trajectory")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis("equal")
    plt.tight_layout()
    plt.savefig("figure_eight_drone_tracking.png", dpi=150)
    print("Saved figure_eight_drone_tracking.png")
